#  Diabetes Data Analysis Project


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [2]:
df = pd.read_csv("../Data/raw_kaggle_diabetes.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../Data/raw_kaggle_diabetes.csv'

## Basic Information

In [ ]:
df.info()
df.describe()

## Check Missing Values

df.isnull().sum()

## Handle Invalid Zero Values (Preprocessing)

In [ ]:
cols = ["Glucose", "BMI", "BloodPressure", "Insulin"]

for col in cols:
    df[col] = df[col].replace(0, np.nan)

# Fill missing values with median
df.fillna(df.median(), inplace=True)

## Create New Features (Feature Engineering)

## Age Group

In [ ]:
df['Age_Group'] = pd.cut(df['Age'],
                        bins=[20, 30, 40, 50, 60, 100],
                        labels=['20-30', '30-40', '40-50', '50-60', '60+'])

## BMI Category

In [ ]:
df['BMI_Category'] = pd.cut(df['BMI'],
                           bins=[0, 18.5, 24.9, 29.9, 100],
                           labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

# Univariate Analysis

## Distribution of Glucose

In [ ]:
plt.figure()
sns.histplot(df['Glucose'], kde=True)
plt.title("Glucose Distribution")
plt.show()

## Outcome Count

In [ ]:
sns.countplot(x='Outcome', data=df)
plt.title("Diabetic vs Non-Diabetic")
plt.show()

# Bivariate Analysis

## Glucose vs Outcome

In [ ]:
sns.boxplot(x='Outcome', y='Glucose', data=df)
plt.title("Glucose vs Outcome")
plt.show()

## BMI vs Outcome

In [ ]:
sns.boxplot(x='Outcome', y='BMI', data=df)
plt.title("BMI vs Outcome")
plt.show()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Key Insights

- Higher glucose levels are strongly associated with diabetes
- Patients with higher BMI are more likely to be diabetic
- Age plays a moderate role in diabetes risk
- Combination of Glucose + BMI increases probability

## Save Cleaned Dataset

In [ ]:
df.to_csv("cleaned_diabetes.csv", index=False)

## WHO Dataset Quick EDA

### Since the data is already cleaned we just going to perform basic EDA 

In [ ]:
who_df = pd.read_csv("../Data/WHO_dataset.csv")

who_df.head()
who_df.info()

## Rename column for better Understanding

In [ ]:
who_df.rename(columns={
    "Raised fasting blood glucose (>=7.0 mmol/L) (age-standardized estimate) - Sex: both sexes - Age group: 18+  years of age": "Diabetes_Prevalence"
}, inplace=True)

## Trend Plot:

In [ ]:
plt.figure()
for country in ["India", "United States", "United Kingdom"]:
    temp = who_df[who_df['Entity'] == country]
    plt.plot(temp['Year'], temp['Diabetes_Prevalence'], label=country)

plt.legend()
plt.title("Diabetes Trend by Country")
plt.show()

## Key Insights From WHO Dataset:

- Diabetes increased steadily in all three countries from 1980–2015
- India showed the fastest growth, ending with the highest value
- The UK had the slowest rise, while the exact metric remains unclear due to missing y-axis context

# Save cleaned WHO dataset

In [ ]:
who_df.to_csv("../Data/clean_who_dataset.csv", index=False)

## Connect Pima Indians Diabetes Dataset  to PostgreSQL for further analysis

In [ ]:
import os
from sqlalchemy import create_engine

username = os.environ.get('PG_USER', 'postgres')
password = os.environ.get('PG_PASSWORD', '')
host = os.environ.get('PG_HOST', 'localhost')
port = os.environ.get('PG_PORT', '5432')
database = os.environ.get('PG_DB', 'Diabetes_analysis')

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

df.to_sql('diabetes', engine, if_exists='replace', index=False)
print(f'Data loaded into PostgreSQL table: diabetes')

## Connect  WHO global Diabetes Dataset  to PostgreSQL for further analysis

In [ ]:
import os
from sqlalchemy import create_engine

username = os.environ.get('PG_USER', 'postgres')
password = os.environ.get('PG_PASSWORD', '')
host = os.environ.get('PG_HOST', 'localhost')
port = os.environ.get('PG_PORT', '5432')
database = os.environ.get('PG_DB', 'Diabetes_analysis')

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

who_df.to_sql('global_diabetes', engine, if_exists='replace', index=False)
print(f'Data loaded into PostgreSQL table: global_diabetes')